In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import ndcg_score
import time

In [2]:
import sys
import os

project_root = os.path.abspath('..')

if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
from src.core.vacancy_rec_sys import VacancyRecSys
from src.core.tfidf_rec_sys import VacancyRecSysTfIdf

d:\Projects\vacancy recommendations\.venv_win\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
vacancies_df = pd.read_csv('../data/processed/cleaned_vacancies.csv')

ground_truth_df = pd.read_csv("../data/processed/validation/before_deduplication/ground_truth.csv")

In [5]:
rec_sys_e5 = VacancyRecSys()
rec_sys_e5.initialize(initial_df=vacancies_df)

rec_sys_tfidf = VacancyRecSysTfIdf()
rec_sys_tfidf.initialize(initial_df=vacancies_df)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10916.95it/s]


Кэш загружен. Векторов в базе: 47325
Кэш загружен. Векторов в базе: 47325


In [6]:
def evaluate_recsys_model(model, ground_truth_df, top_k=20):
    ndcg_scores = []
    precision_scores = []

    anchors = ground_truth_df["anchor_id"].unique()

    for anchor_id in anchors:
        try:
            response = model.get_recommendations(original_id=anchor_id, top_k=top_k)
            predicted_ids = [item.vacancy_id for item in response.items]
        except ValueError:
            continue

        gt_subset = ground_truth_df[ground_truth_df["anchor_id"] == anchor_id]

        gt_scores = dict(zip(gt_subset["candidate_id"], gt_subset["relevance_score"]))

        y_true = [gt_scores.get(cand_id, 0) for cand_id in predicted_ids]

        y_pred = [item.score for item in response.items]

        if len(y_true) > 1:
            ndcg = ndcg_score([y_true], [y_pred], k=top_k)
            ndcg_scores.append(ndcg)

        hits = sum(1 for score in y_true if score >= 2)
        precision = hits / top_k
        precision_scores.append(precision)

    return {"NDCG@20": np.mean(ndcg_scores), "Precision@20": np.mean(precision_scores)}

In [7]:
e5_score = evaluate_recsys_model(rec_sys_e5, ground_truth_df, top_k=20)
tfidf_score = evaluate_recsys_model(rec_sys_tfidf, ground_truth_df, top_k=20)

In [8]:
def print_score(score: dict, model: str):
    print(f"Модель:  {model}")
    print(f"NDCG@20:       {score['NDCG@20']:.3f}")
    print(f"Precision@20:  {score['Precision@20']:.3f}")
    print("-" * 20)


print_score(e5_score, "e5")
print_score(tfidf_score, "tfidf")

Модель:  e5
NDCG@20:       0.893
Precision@20:  0.491
--------------------
Модель:  tfidf
NDCG@20:       0.870
Precision@20:  0.528
--------------------


In [9]:
def calculate_catalog_coverage(model, anchor_ids, total_catalog_size, top_k=20):
    unique_recommended_items = set()
    
    for anchor_id in anchor_ids:
        try:
            response = model.get_recommendations(original_id=anchor_id, top_k=top_k)
            predicted_ids = [item.vacancy_id for item in response.items]
            unique_recommended_items.update(predicted_ids)
        except ValueError:
            continue
            
    coverage_score = len(unique_recommended_items) / total_catalog_size
    
    return {
        "Coverage": coverage_score,
        "Unique_Items_Found": len(unique_recommended_items),
        "Total_Items_In_Catalog": total_catalog_size
    }

In [11]:
catalog_size = vacancies_df.shape[0]
n_count = 1000
k = 20

sample_anchors = vacancies_df['vacancy_id'].sample(n=n_count, random_state=42).tolist()

models_to_evaluate = {
    "E5": rec_sys_e5,
    "TF-IDF": rec_sys_tfidf
}

for model_name, model_instance in models_to_evaluate.items():
    print(f"--- Результаты для {model_name} ---")
    
    start_time = time.perf_counter()
    
    coverage_results = calculate_catalog_coverage(
        model=model_instance, 
        anchor_ids=sample_anchors, 
        total_catalog_size=catalog_size, 
        top_k=k
    )
    
    end_time = time.perf_counter()
    
    total_time = end_time - start_time
    latency_per_query_ms = (total_time / n_count) * 1000
    
    print(f"Общее время: {total_time:.2f} сек.")
    print(f"Среднее время (Latency): {latency_per_query_ms:.2f} мс/запрос")
    print(f"Coverage: {coverage_results['Coverage'] * 100:.2f}%")
    print(f"Уникальных вакансий: {coverage_results['Unique_Items_Found']} из {coverage_results['Total_Items_In_Catalog']}\n")

--- Результаты для E5 ---
Общее время: 3.02 сек.
Среднее время (Latency): 3.02 мс/запрос
Coverage: 29.06%
Уникальных вакансий: 13755 из 47325

--- Результаты для TF-IDF ---
Общее время: 124.21 сек.
Среднее время (Latency): 124.21 мс/запрос
Coverage: 27.56%
Уникальных вакансий: 13043 из 47325

